# Reproduce functional neural modules in 2D MiniGrid

**Paper:** [Inducing, Detecting and Characterising Neural Modules](https://proceedings.mlr.press/v267/soligo25a.html) (ICML 2025) <cite data-cite="pmlr-v267-soligo25a">(Soligo et al., 2025)</cite>  
**Reference code:** [annasoligo/BIXRL](https://github.com/annasoligo/BIXRL/tree/8990ceba70e471926d6cec366d8ce97d8e02fedf) at `8990ceba70e471926d6cec366d8ce97d8e02fedf`  
**Target:** `Navix-Dynamic-Obstacles-6x6-Custom-Rand-v0`, 2D only  
**Default execution:** bounded synthetic smoke path, not a scientific reproduction  
**Current evidence:** notebook mechanics and public-API integration only; full artifacts, reference agreement, and paper claims are **not evaluated**.

The scientific path is enabled with `XDRL_BIXRL_MODE=full`. It fails closed unless `XDRL_BIXRL_BUNDLE` points to a bundle whose SHA-256 equals `XDRL_BIXRL_BUNDLE_SHA256`, and `XDRL_CODE_REVISION` names the exact XDRL revision. The bundle is intentionally external to Git and must contain a trained state dict, frozen evaluation observations/actions, training metadata, and reference metrics.


## Method and scope

The target metric is the paper's **correlation-alignment ARI** between the structural and activation-correlation partitions. Agreement means an absolute difference of at most `0.05` from the checksummed reference implementation. Axis structure is reported separately from correlations with the bundle's declared x/y features. The selected module is the module with the largest x-axis correlation; the comparison is a deterministic, disjoint, size-matched set of hidden units. Both are evaluated on the same split. Intervention effects include action accuracy, disagreement, and x/y action-frequency changes against the unmodified policy.

Material differences from the reference are explicit: XDRL uses PyTorch, TensorDict, TorchRL module boundaries, and TDHook workflows rather than JAX/Flax/NAVIX; the graph and merge equations are re-expressed in notebook-local PyTorch/NetworkX code; and the smoke path trains a synthetic axis task rather than MiniGrid PPO. Reference agreement is evaluated only for a full bundle produced by the pinned 2D environment and training configuration. No result here supports 3D MiniGrid, Pong, generic modularity, or a causal interpretation.


In [ ]:
import hashlib
import json
import os
import subprocess
from pathlib import Path

import networkx as nx
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import ActivationCaching
from tdhook.weights import Pruning
from tdhook.workflow import Workflow
from xdrl import interpret

REFERENCE_REVISION = "8990ceba70e471926d6cec366d8ce97d8e02fedf"
MODE = os.environ.get("XDRL_BIXRL_MODE", "smoke")
SEED = 6163
FULL_TRAINING_CONFIG = {
    "algorithm": "PPO",
    "seed": SEED,
    "training_frames": 4_000_000,
    "finetuning_frames": 2_000_000,
    "num_envs": 16,
    "rollout_steps": 128,
    "epochs": 16,
    "minibatches": 8,
    "learning_rate": 0.0005,
    "discount": 0.99,
    "gae_lambda": 0.99,
    "clip_epsilon": 0.2,
    "entropy_coefficient": 0.01,
    "max_grad_norm": 0.5,
    "hidden_size": 32,
    "hidden_layers": 2,
    "connection_cost": 0.02,
    "distance_subtraction": 0.95,
    "swap_frequency": 2,
    "regularization_warmup": [0.2, 0.3],
    "prune_fraction": 0.01,
}
AGREEMENT_TOLERANCE = 0.05
assert MODE in {"smoke", "full"}
torch.manual_seed(SEED)

In [ ]:
def bytes_digest(value):
    return hashlib.sha256(value).hexdigest()


def tensor_digest(value):
    tensor = value.detach().cpu().contiguous()
    header = f"{tensor.dtype}:{tuple(tensor.shape)}:".encode()
    return bytes_digest(header + tensor.numpy().tobytes())


def module_digest(module):
    payload = "".join(f"{key}:{tensor_digest(value)}" for key, value in module.state_dict().items())
    return bytes_digest(payload.encode())


def evaluation_split_digest(observations, actions):
    return bytes_digest((tensor_digest(observations) + tensor_digest(actions.long())).encode())


def smoke_bundle():
    generator = torch.Generator().manual_seed(SEED)
    observations = torch.rand(384, 2, generator=generator) * 2 - 1
    actions = (observations[:, 0] > 0).long() + 2 * (observations[:, 1] > 0).long()
    return {
        "train_observation": observations[:256],
        "train_action": actions[:256],
        "eval_observation": observations[256:],
        "eval_action": actions[256:],
        "reference_correlation_alignment_ari": None,
        "reference_intervention_effect": None,
        "axis_feature_indices": [0, 1],
        "action_axis": ["x", "x", "y", "y"],
        "environment": "synthetic-axis-smoke",
        "training": {"steps": 120, "optimizer": "Adam", "lr": 0.03},
    }


if MODE == "full":
    bundle_path = Path(os.environ["XDRL_BIXRL_BUNDLE"])
    expected_bundle_sha = os.environ["XDRL_BIXRL_BUNDLE_SHA256"].lower()
    code_revision = os.environ["XDRL_CODE_REVISION"]
    running_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    if code_revision != running_revision:
        raise RuntimeError(f"XDRL revision mismatch: declared {code_revision}, running {running_revision}")
    if subprocess.check_output(["git", "status", "--porcelain"], text=True):
        raise RuntimeError("full mode requires a clean XDRL working tree")
    actual_bundle_sha = bytes_digest(bundle_path.read_bytes())
    if actual_bundle_sha != expected_bundle_sha:
        raise RuntimeError(f"full bundle digest mismatch: expected {expected_bundle_sha}, got {actual_bundle_sha}")
    bundle = torch.load(bundle_path, map_location="cpu", weights_only=True)
    required = {
        "state_dict",
        "eval_observation",
        "eval_action",
        "reference_correlation_alignment_ari",
        "reference_intervention_effect",
        "axis_feature_indices",
        "action_axis",
        "environment",
        "training",
    }
    if set(bundle) != required or bundle["environment"] != "Navix-Dynamic-Obstacles-6x6-Custom-Rand-v0":
        raise RuntimeError("full bundle has an unknown schema or is not the pinned 2D environment")
    if len(bundle["axis_feature_indices"]) != 2 or bundle["action_axis"] != ["x", "x", "y", "y"]:
        raise RuntimeError("full bundle must pin x/y feature indices and left/right/up/down action ordering")
    if bundle["training"] != FULL_TRAINING_CONFIG:
        raise RuntimeError("full bundle does not match the pinned PPO training configuration")
else:
    bundle = smoke_bundle()
    actual_bundle_sha = bytes_digest(b"synthetic-smoke-bundle-v1")
    code_revision = "working-tree-smoke"

{
    "mode": MODE,
    "environment": bundle["environment"],
    "bundle_sha256": actual_bundle_sha,
    "training": bundle["training"],
    "claim_ready": False,
}

## Load or induce the sparse, local policy

A full bundle carries the frozen 32-by-32 checkpoint produced by the pinned reference-derived PPO campaign. The smoke path alone optimizes a smaller policy with logarithmic sparsity and a fixed neuron-distance connection cost, then applies a deterministic two-axis projection so module plumbing cannot pass accidentally on an unstructured network. That projection is not the paper's induction method; the output remains maintenance evidence, never a MiniGrid result.


In [ ]:
input_dim = int(bundle["eval_observation"].shape[-1])
hidden_size = 32 if MODE == "full" else 16
action_count = 4
policy_net = torch.nn.Sequential(
    torch.nn.Linear(input_dim, hidden_size),
    torch.nn.Tanh(),
    torch.nn.Linear(hidden_size, hidden_size),
    torch.nn.Tanh(),
    torch.nn.Linear(hidden_size, action_count, bias=False),
)


def locality_penalty(model):
    total = torch.zeros(())
    line = [
        torch.linspace(-1, 1, input_dim),
        torch.linspace(-1, 1, hidden_size),
        torch.linspace(-1, 1, hidden_size),
        torch.linspace(-1, 1, action_count),
    ]
    for layer, left, right in zip((model[0], model[2], model[4]), line, line[1:]):
        distance = (right[:, None] - left[None, :]).abs()
        total = total + torch.log1p(layer.weight.abs() * distance).mean()
    return total


if MODE == "smoke":
    optimizer = torch.optim.Adam(policy_net.parameters(), lr=bundle["training"]["lr"])
    for _ in range(bundle["training"]["steps"]):
        logits = policy_net(bundle["train_observation"])
        log_sparsity = sum(torch.log1p(parameter.abs()).mean() for parameter in policy_net.parameters())
        loss = (
            torch.nn.functional.cross_entropy(logits, bundle["train_action"])
            + 2e-3 * log_sparsity
            + 2e-3 * locality_penalty(policy_net)
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    half = hidden_size // 2
    with torch.no_grad():
        policy_net[0].weight[:half, 1] = 0
        policy_net[0].weight[half:, 0] = 0
        policy_net[2].weight[:half, half:] = 0
        policy_net[2].weight[half:, :half] = 0
        policy_net[4].weight[:2, half:] = 0
        policy_net[4].weight[2:, :half] = 0
else:
    policy_net.load_state_dict(bundle["state_dict"], strict=True)

policy_net.eval()
checkpoint_sha = module_digest(policy_net)
sparsity = (
    torch.cat([parameter.detach().flatten() for parameter in policy_net.parameters()])
    .abs()
    .lt(1e-2)
    .float()
    .mean()
    .item()
)
{
    "checkpoint_sha256": checkpoint_sha,
    "near_zero_fraction": sparsity,
    "locality_penalty": locality_penalty(policy_net).item(),
}

## Establish uninstrumented parity and cache activations

The frozen evaluation split is wrapped in TensorDict. Native TorchRL-module output must exactly match the public XDRL/TDHook workflow before module interpretation proceeds.


In [ ]:
eval_batch = TensorDict(
    {"observation": bundle["eval_observation"].float()},
    batch_size=[len(bundle["eval_action"])],
    names=["transition"],
)
policy = TensorDictModule(policy_net, in_keys=["observation"], out_keys=["logits"])
component = interpret(policy)
native_logits = component(eval_batch.clone())["logits"].detach().clone()
cache = component.run(
    Workflow(ActivationCaching("module.3", cache_key=("activation", "hidden"))),
    eval_batch.clone(),
)
instrumented_logits = cache.data["logits"].detach()
torch.testing.assert_close(instrumented_logits, native_logits, rtol=0, atol=0)
activations = cache.data["activation", "hidden", "module.3"].detach()
{
    "parity": True,
    "examples": len(eval_batch),
    "activation_shape": tuple(activations.shape),
    "model_passes": cache.plan.model_passes,
}

## Extended Louvain and axis-related structure

The structural graph contains every input, hidden, and output neuron with absolute policy weights as edges. Internal Louvain initializes non-input communities, then assigns each input to its strongest community. A deterministic adjacent-community merge accepts only improvements in mean structural isolation plus adjusted-rand alignment with a Louvain partition of the all-neuron activation-correlation graph. This is notebook-local paper logic, not an XDRL API.


In [ ]:
def graph_from_similarity(similarity, neighbours=3):
    graph = nx.Graph()
    graph.add_nodes_from(range(similarity.shape[0]))
    values = similarity.abs().clone()
    values.fill_diagonal_(-1)
    for i in range(similarity.shape[0]):
        for j in values[i].topk(min(neighbours, similarity.shape[0] - 1)).indices.tolist():
            weight = float(values[i, j])
            if weight > 1e-8:
                graph.add_edge(i, j, weight=weight)
    return graph


def adjusted_rand(left, right, size):
    labels = []
    for partition in (left, right):
        label = torch.empty(size, dtype=torch.long)
        for index, community in enumerate(partition):
            label[list(community)] = index
        labels.append(label)
    same_left = labels[0][:, None] == labels[0][None, :]
    same_right = labels[1][:, None] == labels[1][None, :]
    upper = torch.triu(torch.ones(size, size, dtype=torch.bool), diagonal=1)
    tp = (same_left & same_right & upper).sum().item()
    fp = (same_left & ~same_right & upper).sum().item()
    fn = (~same_left & same_right & upper).sum().item()
    tn = (~same_left & ~same_right & upper).sum().item()
    denominator = (tp + fp) * (fp + tn) + (tp + fn) * (fn + tn)
    return 0.0 if denominator == 0 else 2 * (tp * tn - fp * fn) / denominator


assert adjusted_rand([{0, 1}, {2, 3, 4}], [{0, 1}, {2, 3, 4}], 5) == 1.0


def isolation(graph, partition):
    scores = []
    for community in partition:
        internal = sum(data["weight"] for u, v, data in graph.edges(data=True) if u in community and v in community)
        external = sum(data["weight"] for u, v, data in graph.edges(data=True) if (u in community) != (v in community))
        scores.append(internal / (internal + external) if internal + external else 0.0)
    return sum(scores) / len(scores)


def internal_louvain(weight_graph, input_nodes):
    internal = weight_graph.subgraph(set(weight_graph) - set(input_nodes))
    partition = [set(group) for group in nx.community.louvain_communities(internal, seed=SEED, weight="weight")]
    for node in input_nodes:
        strengths = [
            sum(weight_graph[node][neighbour]["weight"] for neighbour in weight_graph[node] if neighbour in group)
            for group in partition
        ]
        partition[max(range(len(strengths)), key=strengths.__getitem__)].add(node)
    return partition


def extended_louvain(weight_graph, correlation_partition, initial_partition):
    partition = [set(group) for group in initial_partition]

    def score(groups):
        return isolation(weight_graph, groups) + adjusted_rand(groups, correlation_partition, len(weight_graph))

    while len(partition) > 1:
        current = score(partition)
        best = (current, None)
        for i in range(len(partition)):
            for j in range(i + 1, len(partition)):
                if not any(weight_graph.has_edge(u, v) for u in partition[i] for v in partition[j]):
                    continue
                candidate = [group.copy() for group in partition]
                candidate[i] |= candidate[j]
                del candidate[j]
                best = max(best, (score(candidate), (i, j)), key=lambda item: item[0])
        if best[1] is None or best[0] <= current + 1e-12:
            break
        i, j = best[1]
        partition[i] |= partition[j]
        del partition[j]
    return sorted((set(group) for group in partition), key=lambda group: min(group))


layer_sizes = (input_dim, hidden_size, hidden_size, action_count)
offsets = (0, input_dim, input_dim + hidden_size, input_dim + 2 * hidden_size)
weight_graph = nx.Graph()
weight_graph.add_nodes_from(range(sum(layer_sizes)))
for layer, left_offset, right_offset in zip((policy_net[0], policy_net[2], policy_net[4]), offsets, offsets[1:]):
    for right in range(layer.weight.shape[0]):
        for left in range(layer.weight.shape[1]):
            weight = float(layer.weight[right, left].detach().abs())
            if weight > 1e-8:
                weight_graph.add_edge(left_offset + left, right_offset + right, weight=weight)
with torch.no_grad():
    hidden_one = policy_net[1](policy_net[0](eval_batch["observation"]))
    hidden_two = policy_net[3](policy_net[2](hidden_one))
all_activations = torch.cat((eval_batch["observation"], hidden_one, hidden_two, native_logits), dim=1)
correlation = torch.corrcoef(all_activations.T).nan_to_num()
correlation_graph = graph_from_similarity(correlation)
correlation_partition = [
    set(group) for group in nx.community.louvain_communities(correlation_graph, seed=SEED, weight="weight")
]
initial_partition = internal_louvain(weight_graph, range(input_dim))
modules = extended_louvain(weight_graph, correlation_partition, initial_partition)
correlation_alignment_ari = adjusted_rand(modules, correlation_partition, len(weight_graph))
{
    "modules": modules,
    "correlation_partition": correlation_partition,
    "correlation_alignment_ari": correlation_alignment_ari,
    "isolation": isolation(weight_graph, modules),
}

In [ ]:
coordinates = eval_batch["observation"][:, bundle["axis_feature_indices"]]
axis_rows = []
hidden_two_offset = offsets[2]
for module_index, module in enumerate(modules):
    hidden_units = sorted(
        node - hidden_two_offset for node in module if hidden_two_offset <= node < hidden_two_offset + hidden_size
    )
    if not hidden_units:
        continue
    response = activations[:, hidden_units].mean(dim=1)
    corr = torch.corrcoef(torch.stack((response, coordinates[:, 0], coordinates[:, 1]))).nan_to_num()[0, 1:]
    axis_rows.append(
        {
            "module": module_index,
            "units": hidden_units,
            "x": float(corr[0]),
            "y": float(corr[1]),
            "selectivity": float(corr.abs().max()),
        }
    )
axis_alignment = sum(row["selectivity"] for row in axis_rows) / len(axis_rows)
eligible_rows = [row for row in axis_rows if len(row["units"]) <= hidden_size // 2]
if not eligible_rows:
    raise RuntimeError("no module permits a disjoint size-matched null")
target_row = max(eligible_rows, key=lambda row: abs(row["x"]))
target_units = target_row["units"]
available = [unit for unit in range(hidden_size) if unit not in target_units]
if len(available) < len(target_units):
    raise RuntimeError("cannot construct a disjoint size-matched null")
control_units = available[: len(target_units)]
{
    "axis_alignment": axis_alignment,
    "modules": axis_rows,
    "selected_units": target_units,
    "control_units": control_units,
    "selection_used_eval_actions": False,
}

## Paired direct-weight intervention and control

TDHook pruning is applied to the output weights of the selected hidden units before inference. XDRL runs each matched pair with independent TensorDict inputs, restored module state, declared workflow differences, and typed input/output artifact evidence. XDRL's pair manifest records mechanics; it does not call the difference causal.


In [ ]:
def absolute_importance(*, parameter, **_):
    return parameter.abs()


def target_importance(*, parameter, **_):
    score = torch.ones_like(parameter)
    if parameter.ndim == 2:
        score[:, target_units] = 0
    return score


def control_importance(*, parameter, **_):
    score = torch.ones_like(parameter)
    if parameter.ndim == 2:
        score[:, control_units] = 0
    return score


def paired_pruning(selected, importance):
    baseline_workflow = Workflow(
        Pruning(importance_callback=absolute_importance, amount_to_prune=0, relative_path="module.4")
    )
    intervention_workflow = Workflow(
        Pruning(importance_callback=importance, amount_to_prune=action_count * len(selected), relative_path="module.4")
    )
    torch.manual_seed(SEED)
    baseline = component.run(baseline_workflow, eval_batch.clone())
    torch.manual_seed(SEED)
    intervention = component.run(intervention_workflow, eval_batch.clone())
    return {"baseline": baseline, "intervention": intervention}


target_pair = paired_pruning(target_units, target_importance)
control_pair = paired_pruning(control_units, control_importance)
for pair in (target_pair, control_pair):
    torch.testing.assert_close(pair["baseline"].data["logits"], native_logits, rtol=0, atol=0)
    assert not any(child._forward_hooks for child in policy.modules())

In [ ]:
labels = bundle["eval_action"].long()
baseline_actions = native_logits.argmax(dim=-1)
axis_ids = {
    axis: torch.tensor([index for index, value in enumerate(bundle["action_axis"]) if value == axis])
    for axis in ("x", "y")
}


def axis_frequency(actions, axis):
    return float(torch.isin(actions, axis_ids[axis]).float().mean())


def effect(pair):
    actions = pair["intervention"].data["logits"].argmax(dim=-1)
    return {
        "accuracy": float((actions == labels).float().mean()),
        "accuracy_delta": float((actions == labels).float().mean() - (baseline_actions == labels).float().mean()),
        "action_disagreement": float((actions != baseline_actions).float().mean()),
        "x_action_frequency_delta": axis_frequency(actions, "x") - axis_frequency(baseline_actions, "x"),
        "y_action_frequency_delta": axis_frequency(actions, "y") - axis_frequency(baseline_actions, "y"),
    }


metrics = {
    "mode": MODE,
    "correlation_alignment_ari": correlation_alignment_ari,
    "axis_alignment_diagnostic": axis_alignment,
    "baseline_accuracy": float((baseline_actions == labels).float().mean()),
    "target_intervention": effect(target_pair),
    "size_matched_control": effect(control_pair),
}
reference = bundle["reference_correlation_alignment_ari"]
agreement = None if MODE == "smoke" else abs(correlation_alignment_ari - float(reference)) <= AGREEMENT_TOLERANCE
metrics["reference_correlation_alignment_ari"] = reference
metrics["agreement_within_0.05"] = agreement
metrics

## Results and scope

The cell below deliberately separates maintenance execution from scientific evidence. A rendered smoke notebook is not reference agreement or a paper claim. A full run may report positive, negative, or inconclusive agreement; the notebook retains its metrics and explicit input/output digests either way.


In [ ]:
results_summary = {
    "smoke_execution": "passed",
    "full_artifacts": "verified" if MODE == "full" else "not supplied",
    "reference_agreement": ("passed" if agreement else "failed") if agreement is not None else "not evaluated",
    "paper_claim": "not evaluated; 2D target only",
    "claim_limit": "No inference to 3D MiniGrid, Pong, general modularity, or causality.",
}
if MODE == "full":
    output_root = Path(os.environ["XDRL_BIXRL_OUTPUT_ROOT"])
    output_root.mkdir(parents=True, exist_ok=True)
    evidence = {
        "metrics": metrics,
        "results_summary": results_summary,
        "input_digests": {
            "checkpoint": checkpoint_sha,
            "evaluation_split": evaluation_split_digest(eval_batch["observation"], bundle["eval_action"]),
            "reference_bundle": actual_bundle_sha,
        },
        "output_digests": {
            "target": tensor_digest(target_pair["intervention"].data["logits"]),
            "control": tensor_digest(control_pair["intervention"].data["logits"]),
        },
    }
    payload = json.dumps(evidence, indent=2, sort_keys=True) + "\n"
    (output_root / "bixrl-functional-modularity-evidence.json").write_text(payload)
    results_summary["evidence_sha256"] = bytes_digest(payload.encode())
results_summary